# Jaccard Similarity 

In this section you will construct a similarity metric based on the Jaccard similarity coefficient. 

Remember sets from your mathematics class? Well the coefficient is rather simple as it is based on sets operations, namely intersection and union.

## 1. Load the dataset

In [8]:
# code goes here
import pandas as pd 
df = pd.read_csv('data/BX-Book-Ratings-Filtered.csv', sep=';', encoding='latin-1')
df

,User-ID,ISBN,Book-Rating
0,4017,006000438X,10
1,4017,0060915544,8
2,4017,0060929871,9
3,4017,0060930187,10
4,4017,0060964049,9
...,...,...,...
11360,276050,0553377868,7
11361,276050,0671021001,9
11362,276050,067102423X,8
11363,276050,0679746048,7


## 2. Make a set for each book entry

Now we need to construct a set for each book in our dataset. This set will be composed of User-IDs who rated the book. 

From our data frame, can you construct a python dictionary containing ISBNs as keys and an array of User-IDs as values? 

In [7]:
# code goes here
book_user_dict = df.groupby("ISBN")["User-ID"].apply(list).to_dict()

print(list(book_user_dict.items())[:5]) 

[('002542730X', [11676, 52584, 110934, 113270, 128835, 150979, 179734, 208671, 225763, 229741]), ('006000438X', [4017, 6242, 6575, 8454, 11676, 17003, 89602, 115435, 149908, 224349]), ('0060096195', [7346, 8067, 13552, 25409, 95359, 181687, 251422]), ('006016848X', [11676, 16634, 39281, 165308, 236340]), ('0060173289', [29526, 86189, 110912, 133747, 137589, 141710, 276050])]


## 3. Jaccard distance function

Here is the `jaccard_distance` function we provide you for the exercise. It calculates the distance between 2 books, taking into account who rated them (i.e., if more users rated the same book, then the books are closer). 

Please have a closer look at the function. As you can see, we are using python sets and the function is expecting two arrays composed of User-IDs.

In [14]:
def jaccard_distance(user_ids_isbn_a, user_ids_isbn_b):
    set_a, set_b = set(user_ids_isbn_a), set(user_ids_isbn_b)
    intersection = set_a & set_b  # Users who rated both books
    union = set_a | set_b  # Users who rated at least one of the books

    if not union:
        return 0  # Avoid division by zero, return 0 if both sets are empty

    return len(intersection) / float(len(union))


## 4. Calculate distances 

Here is the ISBN of a book in our dataset (you can of course choose another one!). 

Can you calculate this book's jaccard distance from all the other books in the dataset?

In [15]:
a_book_isbn = '002542730X'

# code goes here
jaccard_distances = {}

# Iterate over all books except the target book
for isbn, users in book_user_dict.items():
    if isbn != a_book_isbn:
        jaccard_distances[isbn] = jaccard_distance(book_user_dict[a_book_isbn], users)

# Convert to DataFrame for better readability
jaccard_df = pd.DataFrame(jaccard_distances.items(), columns=["ISBN", "Jaccard Distance"])
jaccard_df = jaccard_df.sort_values(by="Jaccard Distance", ascending=False)

# Display top 10 most similar books
print(jaccard_df.head(10))

           ISBN  Jaccard Distance
38   0064407667          0.200000
380  044022103X          0.200000
39   0064407675          0.190476
776  0804108749          0.187500
231  0375756981          0.176471
40   0064407683          0.176471
432  0446527785          0.166667
804  0894805770          0.153846
144  0330367358          0.153846
768  0786885688          0.150000


## 5. Function calculating distances 

Considering the code above, can you make a function that will take as input a given book's ISBN and calculate its distance from all other books in our dataset? 

In [17]:
# code goes here

def calculate_jaccard_distances(target_isbn, book_user_dict):
    """
    Calculate Jaccard distances between a given book and all other books in the dataset.

    Parameters:
    - target_isbn (str): The ISBN of the book for which to calculate distances.
    - book_user_dict (dict): Dictionary with ISBNs as keys and lists of User-IDs as values.

    Returns:
    - pd.DataFrame: A sorted DataFrame with ISBNs and their corresponding Jaccard distances.
    """
    if target_isbn not in book_user_dict:
        print(f"Book {target_isbn} not found in dataset.")
        return None

    # Compute Jaccard distances
    distances = {
        isbn: jaccard_distance(book_user_dict[target_isbn], users)
        for isbn, users in book_user_dict.items() if isbn != target_isbn
    }

    # Convert results to DataFrame and sort by similarity
    jaccard_df = pd.DataFrame(distances.items(), columns=["ISBN", "Jaccard Distance"])
    jaccard_df = jaccard_df.sort_values(by="Jaccard Distance", ascending=False)

    return jaccard_df
